Information Retrieval System 
- The system process text dataset about a NewsPaper using different methods and algorithms using TF-IDF.    

In [15]:
#Importing all needed libraries 
import csv
import pickle
import tkinter as tk
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import precision_score, recall_score, f1_score

#Processing data through numerous methodologies.
class InvertedIndex:
    def __init__(self, csv_file=None):
        self.csv_file = csv_file
        self.docs = []
        self.labels = []
        self.lines = 0
        self.inverted_index = {}
        self.indexer = {}
        self.vectorizer = None
        self.tfidf_matrix = None
        self.classifier = None

        if self.csv_file:
            self._read_csv()

# evaluating retrieval of certain queries by calculating precision, recall and F1 score.
    def evaluate_retrieval(self, queries, true_labels):
        """
        Evaluate the performance of the retrieval system using precision, recall, and F1 score.
        """
        all_precision = []
        all_recall = []
        all_f1 = []

        for query, true_set in zip(queries, true_labels):
            results = self.search(query)
            retrieved_docs = [doc for doc, score in results if score > 0.0]
            
            # Generate predicted labels (binary: relevant or not)
            predicted_labels = [1 if i in true_set else 0 for i in range(len(self.docs))]

            # Generate true labels (binary: relevant or not)
            true_labels_binary = [1 if i in true_set else 0 for i in range(len(self.docs))]

            precision = precision_score(true_labels_binary, predicted_labels)
            recall = recall_score(true_labels_binary, predicted_labels)
            f1 = f1_score(true_labels_binary, predicted_labels)

            all_precision.append(precision)
            all_recall.append(recall)
            all_f1.append(f1)

        avg_precision = sum(all_precision) / len(all_precision)
        avg_recall = sum(all_recall) / len(all_recall)
        avg_f1 = sum(all_f1) / len(all_f1)

        return avg_precision, avg_recall, avg_f1
    
# function to read the documents.
    def _read_csv(self):
        with open(self.csv_file, 'r') as file:
            reader = csv.reader(file)
            for row in reader:
                if len(row) >= 2: 
                    self.docs.append(row[0])
                    self.labels.append(row[1])
                    self.lines += 1
                    
# functions to return information about the documents. 
    def Display_docs(self):
        return self.docs

    def Number_of_docs(self):
        return self.lines

    def Split_docs(self):
        return self.docs

# Tokenizing the documents.
    def Tokenization(self):
        tokens_list = []
        for doc in self.docs:
            tokens_list.append(doc.lower().split())
        return tokens_list
    
# removing all the stop words.
    def Stopping(self):
        with open('stopwords.txt', 'r') as file:
            stop_words = file.read().split()

        NewList = []
        tokens_list = self.Tokenization()
        for doc in tokens_list:
            NewDoc = ["".join(ch for ch in word if ch.isalnum()) for word in doc]
            NewList.append(NewDoc)

        after_stopping = [[token for token in doc if token not in stop_words] for doc in NewList]
        return after_stopping
    
# function to process the data using inverted index method.
    def Inverted_Index(self):
        if not self.inverted_index:
            after_stopping = self.Stopping()
            for i, doc in enumerate(after_stopping):
                for token in doc:
                    if token in self.docs[i]:
                        if token in self.inverted_index:
                            self.inverted_index[token].add(i + 1)
                        else:
                            self.inverted_index[token] = {i + 1}
        return self.inverted_index

    def Indexer(self):
        if not self.indexer:
            after_stopping = self.Stopping()
            for i, doc in enumerate(after_stopping):
                for token in doc:
                    self.indexer[token] = i
        return self.indexer

# calculating frequency in each term.
    def Term_squences(self):
        indexer = self.Indexer()
        return sorted(indexer.items(), key=lambda x: x[1])

# sorting the terms.
    def Sorting_Alphabetically(self):
        indexer = self.Indexer()
        return sorted(indexer.items(), key=lambda x: x[0])

    
# saving the file after it has been processed.
    def save_index(self, file_path):
        data = {
            'docs': self.docs,
            'labels': self.labels,
            'lines': self.lines,
            'inverted_index': self.inverted_index,
            'indexer': self.indexer
        }
        with open(file_path, 'wb') as file:
            pickle.dump(data, file)
# loading the file after it has been processed.
    def load_index(self, file_path):
        with open(file_path, 'rb') as file:
            data = pickle.load(file)
            self.docs = data['docs']
            self.labels = data['labels']
            self.lines = data['lines']
            self.inverted_index = data['inverted_index']
            self.indexer = data['indexer']

# TF-IDF.
    def build_tfidf_matrix(self):
        self.vectorizer = TfidfVectorizer(stop_words='english')
        self.tfidf_matrix = self.vectorizer.fit_transform(self.docs)

    def train_classifier(self):
        if self.tfidf_matrix is None:
            self.build_tfidf_matrix()
        
        X_train, X_test, y_train, y_test = train_test_split(self.tfidf_matrix, self.labels, test_size=0.2, random_state=42)
        self.classifier = MultinomialNB()
        self.classifier.fit(X_train, y_train)
        accuracy = self.classifier.score(X_test, y_test)
        print(f'Classifier accuracy: {accuracy:.2f}')

    def classify_document(self, doc):
        if self.vectorizer is None or self.classifier is None:
            raise ValueError("The vectorizer and classifier must be initialized and trained before classification.")
        doc_tfidf = self.vectorizer.transform([doc])
        return self.classifier.predict(doc_tfidf)

# function to search the query in the processed document.
    def search(self, query):
        # Ensure the TF-IDF matrix is built
        if self.tfidf_matrix is None:
            self.build_tfidf_matrix()
        
        # Preprocess and vectorize the query
        query_tfidf = self.vectorizer.transform([query])
        # Calculate cosine similarities between the query and all documents
        cosine_similarities = cosine_similarity(query_tfidf, self.tfidf_matrix).flatten()
        # Rank documents by similarity score
        ranked_doc_indices = cosine_similarities.argsort()[::-1]
        return [(self.docs[i], cosine_similarities[i]) for i in ranked_doc_indices if cosine_similarities[i] > 0.0]
    
    

# GUI ask for the user input of the query and return back the documents where it was found.
class InvertedIndexGUI:
    def __init__(self, master, inverted_index):
        self.master = master
        self.inverted_index = inverted_index
        self.master.title("Inverted Index Search")
        
        self.label = tk.Label(master, text="Enter query:")
        self.label.pack()

        self.query_entry = tk.Entry(master, width=50)
        self.query_entry.pack()

        self.search_button = tk.Button(master, text="Search", command=self.search)
        self.search_button.pack()

        self.result_text = tk.Text(master, height=15, width=100)
        self.result_text.pack()

    def search(self):
        query = self.query_entry.get()
        results = self.inverted_index.search(query)

        self.result_text.delete(1.0, tk.END)
        self.result_text.insert(tk.END, f"Found in {len(results)} documents:\n\n")
        for doc_index, score in results:
         if score > 0.0:
            self.result_text.insert(tk.END, f"Document: {doc_index} (Score: {score:.2f})\n")

if __name__ == "__main__":
    csv_file = "C:/Users/Magic Systems/OneDrive/Desktop/NewspaperDataset/alt.atheism.csv"
    inverted_index = InvertedIndex(csv_file)
    inverted_index.build_tfidf_matrix()
    inverted_index.train_classifier()

    root = tk.Tk()
    app = InvertedIndexGUI(root, inverted_index)
    root.mainloop()

Classifier accuracy: 0.09


In [16]:
#specifying the document file path
csv_file = "C:/Users/Magic Systems/OneDrive/Desktop/NewspaperDataset/alt.atheism.csv"
InvIndex = InvertedIndex(csv_file)
InvIndex.Display_docs()

['Write to:  FFRF',
 'Evolution Designs sell the "Darwin fish".  It\'s a fish symbol',
 'Christians stick on their cars',
 'Write to:  Evolution Designs',
 'try mailing <figmo@netcom.com>.  For net people who go to Lynn directly',
 'AAP publish various atheist books -- critiques of the Bible',
 'Biblical contradictions',
 '372 pp.  ISBN 0-910309-26-4',
 'absurdities',
 'Contradicts Itself"',
 'Write to:  American Atheist Press',
 '      or:  7215 Cameron Road',
 'Write to:  700 East Amherst Street',
 'Prometheus Books',
 'black freethought.  They publish a quarterly newsletter',
 'Write to:  Norm R. Allen',
 '           Buffalo',
 'The National Secular Society publish "The Freethinker"',
 'Postfach 880',
 'MIZ-Vertrieb',
 'For atheist books',
 'IBDK',
 'Postfach 3005',
 'events are fictitious.  Any similarity to living or dead gods -- uh',
 'WALTER M. MILLER',
 'copying blueprints from "Saint Leibowitz"',
 'Post atomic doomsday novel set in clerical states.  The church',
 'forbids that

- Printing out the details of the document:

In [17]:
InvIndex.Number_of_docs()

29661

In [18]:
InvIndex.Split_docs()

['Write to:  FFRF',
 'Evolution Designs sell the "Darwin fish".  It\'s a fish symbol',
 'Christians stick on their cars',
 'Write to:  Evolution Designs',
 'try mailing <figmo@netcom.com>.  For net people who go to Lynn directly',
 'AAP publish various atheist books -- critiques of the Bible',
 'Biblical contradictions',
 '372 pp.  ISBN 0-910309-26-4',
 'absurdities',
 'Contradicts Itself"',
 'Write to:  American Atheist Press',
 '      or:  7215 Cameron Road',
 'Write to:  700 East Amherst Street',
 'Prometheus Books',
 'black freethought.  They publish a quarterly newsletter',
 'Write to:  Norm R. Allen',
 '           Buffalo',
 'The National Secular Society publish "The Freethinker"',
 'Postfach 880',
 'MIZ-Vertrieb',
 'For atheist books',
 'IBDK',
 'Postfach 3005',
 'events are fictitious.  Any similarity to living or dead gods -- uh',
 'WALTER M. MILLER',
 'copying blueprints from "Saint Leibowitz"',
 'Post atomic doomsday novel set in clerical states.  The church',
 'forbids that

In [19]:
InvIndex.Tokenization()

[['write', 'to:', 'ffrf'],
 ['evolution',
  'designs',
  'sell',
  'the',
  '"darwin',
  'fish".',
  "it's",
  'a',
  'fish',
  'symbol'],
 ['christians', 'stick', 'on', 'their', 'cars'],
 ['write', 'to:', 'evolution', 'designs'],
 ['try',
  'mailing',
  '<figmo@netcom.com>.',
  'for',
  'net',
  'people',
  'who',
  'go',
  'to',
  'lynn',
  'directly'],
 ['aap',
  'publish',
  'various',
  'atheist',
  'books',
  '--',
  'critiques',
  'of',
  'the',
  'bible'],
 ['biblical', 'contradictions'],
 ['372', 'pp.', 'isbn', '0-910309-26-4'],
 ['absurdities'],
 ['contradicts', 'itself"'],
 ['write', 'to:', 'american', 'atheist', 'press'],
 ['or:', '7215', 'cameron', 'road'],
 ['write', 'to:', '700', 'east', 'amherst', 'street'],
 ['prometheus', 'books'],
 ['black', 'freethought.', 'they', 'publish', 'a', 'quarterly', 'newsletter'],
 ['write', 'to:', 'norm', 'r.', 'allen'],
 ['buffalo'],
 ['the', 'national', 'secular', 'society', 'publish', '"the', 'freethinker"'],
 ['postfach', '880'],
 ['m

- Removing stopping words results:

In [20]:
InvIndex.Stopping()

[['write', 'ffrf'],
 ['evolution', 'designs', 'sell', 'darwin', 'fish', 'fish', 'symbol'],
 ['christians', 'stick', 'cars'],
 ['write', 'evolution', 'designs'],
 ['try', 'mailing', 'figmonetcomcom', 'net', 'people', 'lynn', 'directly'],
 ['aap', 'publish', 'various', 'atheist', 'books', '', 'critiques', 'bible'],
 ['biblical', 'contradictions'],
 ['372', 'pp', 'isbn', '0910309264'],
 ['absurdities'],
 ['contradicts'],
 ['write', 'american', 'atheist', 'press'],
 ['7215', 'cameron', 'road'],
 ['write', '700', 'east', 'amherst', 'street'],
 ['prometheus', 'books'],
 ['black', 'freethought', 'publish', 'quarterly', 'newsletter'],
 ['write', 'norm', 'r', 'allen'],
 ['buffalo'],
 ['national', 'secular', 'society', 'publish', 'freethinker'],
 ['postfach', '880'],
 ['mizvertrieb'],
 ['atheist', 'books'],
 ['ibdk'],
 ['postfach', '3005'],
 ['events', 'fictitious', 'similarity', 'living', 'dead', 'gods', '', 'uh'],
 ['walter', 'm', 'miller'],
 ['copying', 'blueprints', 'saint', 'leibowitz'],
 [

- Inverted Index Results:

In [21]:
InvIndex.Inverted_Index()

{'sell': {2,
  4980,
  6151,
  6394,
  7578,
  8074,
  8819,
  9889,
  14867,
  16038,
  16281,
  17465,
  17961,
  18706,
  19776,
  24754,
  25925,
  26168,
  27352,
  27848,
  28593},
 'fish': {2,
  1011,
  8819,
  9836,
  9889,
  10898,
  18706,
  19723,
  19776,
  20785,
  28593,
  29610},
 'symbol': {2, 2739, 8819, 9889, 12626, 18706, 19776, 22513, 28593},
 'stick': {3,
  1485,
  6113,
  6913,
  8820,
  9890,
  11372,
  16000,
  16800,
  18707,
  19777,
  21259,
  25887,
  26687,
  28594},
 'cars': {3, 936, 8820, 9890, 10823, 18707, 19777, 20710, 28594},
 'try': {5,
  1291,
  1358,
  2453,
  2799,
  3778,
  3907,
  5913,
  8475,
  8822,
  8903,
  9892,
  11178,
  11245,
  12340,
  12686,
  13665,
  13794,
  15800,
  18362,
  18709,
  18790,
  19779,
  21065,
  21132,
  22227,
  22573,
  23552,
  23681,
  25687,
  28249,
  28596,
  28677},
 'mailing': {5, 8822, 9892, 18709, 19779, 28596},
 'net': {5,
  2798,
  4885,
  8257,
  8822,
  8902,
  9892,
  12685,
  14772,
  18144,
  1870

In [22]:
InvIndex.Indexer()

{'write': 28608,
 'ffrf': 28591,
 'evolution': 29008,
 'designs': 28594,
 'sell': 28592,
 'darwin': 28592,
 'fish': 29609,
 'symbol': 28592,
 'christians': 28593,
 'stick': 28593,
 'cars': 28593,
 'try': 28676,
 'mailing': 28595,
 'figmonetcomcom': 28595,
 'net': 28675,
 'people': 29629,
 'lynn': 28595,
 'directly': 28595,
 'aap': 28598,
 'publish': 28610,
 'various': 29491,
 'atheist': 29297,
 'books': 29655,
 '': 29639,
 'critiques': 28598,
 'bible': 29602,
 'biblical': 28599,
 'contradictions': 28599,
 '372': 28600,
 'pp': 28600,
 'isbn': 28600,
 '0910309264': 28600,
 'absurdities': 28601,
 'contradicts': 28602,
 'american': 28646,
 'press': 28603,
 '7215': 28604,
 'cameron': 28604,
 'road': 28604,
 '700': 28605,
 'east': 28605,
 'amherst': 28605,
 'street': 29091,
 'prometheus': 28606,
 'black': 28607,
 'freethought': 28607,
 'quarterly': 28607,
 'newsletter': 28607,
 'norm': 28666,
 'r': 28666,
 'allen': 28666,
 'buffalo': 28609,
 'national': 28610,
 'secular': 28610,
 'society': 

- Frequency of each term:

In [23]:
InvIndex.Term_squences()

[('neutral', 19853),
 ('cohesion', 19854),
 ('progression', 19854),
 ('slippery', 19862),
 ('agnosticism', 19864),
 ('beware', 19865),
 ('scepticism', 19869),
 ('provably', 19876),
 ('primes', 19878),
 ('interacts', 19885),
 ('nondetectable', 19886),
 ('verifiable', 19889),
 ('presence', 19890),
 ('records', 19892),
 ('tendency', 19905),
 ('runs', 19909),
 ('antireligion', 19910),
 ('questioned', 19911),
 ('antireligious', 19914),
 ('encroaching', 19918),
 ('legislate', 19919),
 ('encouraging', 19922),
 ('reforms', 19922),
 ('campaigns', 19923),
 ('campaign', 19923),
 ('discriminate', 19924),
 ('voters', 19925),
 ('lawmakers', 19925),
 ('intimidating', 19927),
 ('nonchristians', 19927),
 ('wasting', 19929),
 ('contribute', 19930),
 ('charities', 19930),
 ('selfrighteous', 19933),
 ('upbringing', 19937),
 ('backlash', 19940),
 ('conformism', 19940),
 ('heavy', 19941),
 ('metal', 19941),
 ('verdi', 19942),
 ('requiem', 19942),
 ('dress', 19943),
 ('sinners', 19953),
 ('jeffrey', 19955),


In [24]:
InvIndex.Sorting_Alphabetically()

[('', 29639),
 ('0', 28393),
 ('013423tan102psuvmpsuedu', 24985),
 ('051018', 25492),
 ('0910309264', 28600),
 ('0930', 25050),
 ('1', 29271),
 ('10', 24216),
 ('100', 26164),
 ('110', 26477),
 ('111', 26265),
 ('111213', 26521),
 ('112430', 21237),
 ('114127buedu', 20885),
 ('115288buedu', 27187),
 ('115437buedu', 27192),
 ('115468buedu', 23054),
 ('115561buedu', 26805),
 ('115565buedu', 29067),
 ('115621buedu', 28179),
 ('115694buedu', 27391),
 ('115793buedu', 25282),
 ('116171buedu', 27381),
 ('116533buedu', 27223),
 ('11838viceicotekcom', 25740),
 ('11853viceicotekcom', 28770),
 ('11862viceicotekcom', 25543),
 ('11867viceicotekcom', 25549),
 ('12', 28217),
 ('1200', 22238),
 ('12122', 26232),
 ('1219', 23387),
 ('125', 29606),
 ('126', 29608),
 ('13', 23942),
 ('130', 24090),
 ('1330', 29592),
 ('1338', 29524),
 ('133834', 28769),
 ('143048io30436mainemaineedu', 20228),
 ('15', 24729),
 ('1527', 26613),
 ('154', 22457),
 ('16', 26591),
 ('161', 29552),
 ('1618', 26576),
 ('165', 29

In [25]:
InvIndex.save_index('indexed_dataTF.pkl')
# Loading the index from a file
LoadedInvIndex = InvertedIndex()
LoadedInvIndex.load_index('indexed_dataTF.pkl')

- Training Classifier

In [26]:
InvIndex.train_classifier()
doc = "example document to classify"
print(f'Document classification: {InvIndex.classify_document(doc)}')

Classifier accuracy: 0.09
Document classification: ['']


- Testing the search for a query:

In [27]:

query = "I replied to this query"
results = InvIndex.search(query)
for doc, score in results:
    if score > 0.0:
     print(f'Document: {doc}, Score: {score}')



Document: I replied to this query via e-mail, Score: 0.8566454186732855
Document: I replied to this query via e-mail, Score: 0.8566454186732855
Document: I replied to this query via e-mail, Score: 0.8566454186732855
Document: Jesus replied as we might expect Matthew to have done, Score: 0.42568000061172084
Document: Jesus replied as we might expect Matthew to have done, Score: 0.42568000061172084
Document: Jesus replied as we might expect Matthew to have done, Score: 0.42568000061172084


- Evaluation metrics calculated for a certain queries:

In [28]:
queries = ["Subject: Alt.Atheism FAQ: Atheist Resource ", "Can you explain what"]
true_labels = [{453}, {4}]  # Example true labels

# Evaluate the retrieval system
avg_precision, avg_recall, avg_f1 = InvIndex.evaluate_retrieval(queries, true_labels)
print(f"Retrieval System Metrics:")
print(f"Average Precision: {avg_precision:.2f}")
print(f"Average Recall: {avg_recall:.2f}")
print(f"Average F1 Score: {avg_f1:.2f}")

Retrieval System Metrics:
Average Precision: 1.00
Average Recall: 1.00
Average F1 Score: 1.00


Made by:
- Ghazal Almidana
- Ayat Omar
- Zaid Eltabatebi